# Build a mini agent from scratch

**Goal:** understand how a model chooses and uses Python tools to answer a question.

Our agent has two tools: `add_numbers` for arithmetic and `city_distance` for geography. The geography tool gets city coordinates from a public API, then calculates the distance in Python.

**Question → model requests a tool → Python runs it → result goes back to the model → final answer**

An agent combines a **model**, **instructions**, **tools**, and a **loop**. The model selects a tool and its arguments; our code executes the function. We keep the conversation in a list called `messages`. Both tools use the same small loop.

At the end, build a second small agent that compares **heat and tree canopy in two Philadelphia neighborhoods**, using the Responses API and `gpt-5-mini`.


## Step 1: Connect to the model

Use your course environment with `openai` and `requests` installed. If needed, run `%pip install openai requests` in a separate cell, then restart the kernel.

Run the next cell and enter your API key in the **hidden prompt**. If `OPENAI_API_KEY` is already set in the environment that started Jupyter, that value is used instead. Do not put a real key in the notebook's source.

The compression header avoids the Brotli compatibility error encountered earlier. Creating the client does not send an API request; running the demo later does.

In [10]:
import json
import os
from getpass import getpass
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: "),
    default_headers={"Accept-Encoding": "gzip, deflate"},
)
MODEL = "gpt-4o"

## Step 2: Write Python tools

A tool is an ordinary function. `add_numbers` runs entirely locally. `city_distance` calls the [Open-Meteo geocoding API](https://open-meteo.com/en/docs/geocoding-api) to get city coordinates, then uses the **haversine formula** to calculate great-circle distance in kilometers.

This is the shortest distance over a spherical Earth, **not driving distance or an airline route**. The first city match is used and its name is returned so you can check it. Add country names for clarity, such as `Paris, France`. Coordinates come from Open-Meteo / GeoNames; this classroom endpoint requires internet access but no extra API key.

In [11]:
import math
import requests

def add_numbers(a, b):
    """Return the sum of two numbers."""
    return a + b

def city_distance(city1, city2):
    """Look up two city centers and return their great-circle distance in km."""
    def locate(city):
        response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en"}, timeout=20,
        )
        response.raise_for_status()
        matches = response.json().get("results", [])
        if not matches:
            raise ValueError(f"City not found: {city}. Try adding its country.")
        return matches[0]

    start, end = locate(city1), locate(city2)
    lat1, lon1 = map(math.radians, [start["latitude"], start["longitude"]])
    lat2, lon2 = map(math.radians, [end["latitude"], end["longitude"]])
    a = math.sin((lat2-lat1)/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin((lon2-lon1)/2)**2
    km = 2 * 6371 * math.asin(math.sqrt(max(0, min(1, a))))
    return {
        "from": f"{start['name']}, {start.get('admin1', '')}, {start.get('country', '')}",
        "to": f"{end['name']}, {end.get('admin1', '')}, {end.get('country', '')}",
        "distance_km": round(km, 1), "type": "great-circle distance",
    }



### Try both tools directly first

These are ordinary Python calls: **the model is not involved yet**. Run addition first, then geography. The geography call needs internet access but does not use your OpenAI API quota. You can run Step 2 on its own without connecting to the model.


In [12]:
print(add_numbers(125, 75))  # Expected: 200


200


In [13]:
# Change these two inputs to explore other cities.
city1 = "Philadelphia, US"
city2 = "New York, US"

distance_result = city_distance(city1, city2)
print(distance_result)
# Expected: matched city names and about 130 km of great-circle distance.


{'from': 'Philadelphia, Pennsylvania, United States', 'to': 'New York, New York, United States', 'distance_km': 129.6, 'type': 'great-circle distance'}


**Pause and inspect:** Which places did the API match? What does `distance_km` measure? Try swapping the cities: should the distance change? Keep this result in mind when you run the geography agent in Step 5.


## Step 3: Describe the tools to the model

The model cannot inspect our local functions. We send a **schema** describing each tool's name, purpose, and inputs.

- `name` matches the Python function.
- `properties` describes each tool's inputs.
- `required` means the listed inputs must be provided.
- `strict` asks for arguments matching this schema.

The `available_tools` dictionary connects each tool name to its actual Python function.

In [14]:
tools = [{
    "type": "function",
    "function": {
        "name": "add_numbers",
        "description": "Add two numbers.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    },
}, {
    "type": "function",
    "function": {
        "name": "city_distance",
        "description": "Look up two cities and calculate great-circle distance in kilometers. Include countries when possible.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {"city1": {"type": "string"}, "city2": {"type": "string"}},
            "required": ["city1", "city2"],
            "additionalProperties": False,
        },
    },
}]
available_tools = {"add_numbers": add_numbers, "city_distance": city_distance}

## Step 4: Write the agent loop

Each turn has four actions:

1. **Ask the model** using the conversation and tool schema.
2. **Check its response.** If no tool is requested, return the final answer.
3. **Run the requested tool** with the JSON arguments.
4. **Append the result** to the conversation, then repeat.

The assistant message records the tool request. The `role="tool"` message records our result; `tool_call_id` connects that result to the request. The three-turn limit prevents the loop from running forever.

This is a minimal classroom example. The loop reports city-lookup errors back to the model. A larger application would add more input validation and API error handling.

In [15]:
def run_agent(question):
    messages = [
        {"role": "system", "content": "Use add_numbers for addition and city_distance for city distances. Report kilometers, the matched cities, and great-circle distance. If a tool returns an error, ask for clarification; do not invent a result. Give a short final answer."},
        {"role": "user", "content": question},
    ]
    for turn in range(1, 4):
        print(f"\n--- Agent turn {turn} ---")
        # 1. Ask the model what to do next.
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        # 2. No tool request means the model has answered.
        if not message.tool_calls:
            return message.content

        # 3. Run each requested Python tool.
        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments)
            print("Tool request:", call.function.name, arguments)
            try:
                result = available_tools[call.function.name](**arguments)
            except (ValueError, requests.RequestException) as error:
                result = {"error": str(error)}
            print("Tool result:", result)

            # 4. Give the tool result back to the model.
            messages.append({
                "role": "tool", "tool_call_id": call.id, "content": json.dumps(result),
            })
    raise RuntimeError("Stopped after three turns without a final answer.")

## Step 5: Let the agent choose a tool

Run Steps 1–4 first. Both demos send real requests to OpenAI and use your API quota. The second also makes two public city-coordinate lookups.

Compare these runs with the direct function calls in Step 2. Here you supply a **question**, and the model supplies the **tool name and arguments**. Read the printed tool request and result before the final answer. Model wording and argument order may vary.


In [16]:
question = "What is 125 + 75?"
answer = run_agent(question)
print("Final answer:", answer)



--- Agent turn 1 ---
Tool request: add_numbers {'a': 125, 'b': 75}
Tool result: 200

--- Agent turn 2 ---
Final answer: The sum of 125 and 75 is 200.


### Geography demo: distance between two cities

The same `run_agent()` function now chooses `city_distance`. No new agent loop is needed. Expect roughly **130 km** between Philadelphia and New York; this refers to the returned city centers, not a road trip.

In [17]:
# Edit the cities in this question, then rerun this cell.
question = "How far apart are Philadelphia, US and New York, US in kilometers?"
answer = run_agent(question)
print("Final answer:", answer)



--- Agent turn 1 ---
Tool request: city_distance {'city1': 'Philadelphia, US', 'city2': 'New York, US'}
Tool result: {'from': 'Philadelphia, Pennsylvania, United States', 'to': 'New York, New York, United States', 'distance_km': 129.6, 'type': 'great-circle distance'}

--- Agent turn 2 ---
Final answer: Philadelphia, Pennsylvania, United States and New York, New York, United States are approximately 129.6 kilometers apart in terms of great-circle distance.


### Read the agent's trace

An addition run should look like this (the final wording can vary):

```text
--- Agent turn 1 ---
Tool request: add_numbers {'a': 125, 'b': 75}
Tool result: 200

--- Agent turn 2 ---
Final answer: 125 + 75 = 200.
```

For geography, look for `Tool request: city_distance`, the two city arguments, and a result with `distance_km` and matched city names. Compare it with your direct tool call. Both paths use the same Python function.

**What makes this an agent?** The model selects an action, our Python loop runs it, and the model uses the returned observation to answer. The model does not execute Python itself. The schema describes the tool; `available_tools` connects that description to executable code. Each call to `run_agent()` starts a fresh conversation.

## Step 6: Explore your first agent

Change one thing at a time. Before each run, predict the tool and arguments; afterward, compare your prediction with the trace.

| Question to try | What to observe |
| --- | --- |
| What is 18.5 + 6.5? | Does the model request `add_numbers` with the right inputs? |
| How far apart are London, United Kingdom and Paris, France? | Check the matched cities and distance in kilometers. |
| Say hello. | Can the model answer without a tool? |
| What are 125 + 75 and 18.5 + 6.5? | Does the model request two tool calls? |
| How far is NotARealCityXYZ from Paris, France? | Does the tool report an error and the model ask for clarification? |

The geography tool returns great-circle distance, not driving distance. Always inspect the matched places: a city name can refer to more than one location.


In [18]:
# Your experiment: replace the question and rerun.
my_question = "How far apart are London, United Kingdom and Paris, France?"
print("Final answer:", run_agent(my_question))



--- Agent turn 1 ---
Tool request: city_distance {'city1': 'London, United Kingdom', 'city2': 'Paris, France'}
Tool result: {'from': 'London, England, United Kingdom', 'to': 'Paris, Île-de-France Region, France', 'distance_km': 343.8, 'type': 'great-circle distance'}

--- Agent turn 2 ---
Final answer: London, England, United Kingdom and Paris, Île-de-France Region, France are approximately 343.8 kilometers apart.


## Step 7: An urban-sustainability agent — neighborhood heat

Now apply the same idea to Philadelphia. Our question is: **How do heat and tree canopy differ between Kensington and Chestnut Hill?**

The values below are **invented classroom data, not measured neighborhood conditions**. `temp_c` means temperature in degrees Celsius; `canopy` means tree-canopy coverage in percent. The table has no measurement date or sampling method, so do not use it to make real-world claims.

This example has the same four parts: **1. Python tool → 2. tool description → 3. model call → 4. loop**. It uses OpenAI's **Responses API**, matching the example in class. The earlier examples use Chat Completions: the loop is the same idea, but the message formats differ. Run Step 1 first to create `client` and import `json`. This demo requires access to `gpt-5-mini` and uses your API quota.

We call the new loop `run_heat_agent` so the earlier `run_agent` examples continue to work.

In [23]:
# 1. TOOL: ordinary Python code and a small sample dataset.
HEAT_DATA = {
    "Kensington": {"temp_c": 38.5, "canopy": 9},
    "Chestnut Hill": {"temp_c": 33.0, "canopy": 48},
}

def get_heat_info(neighborhood):
    """Return classroom sample temperature (C) and tree canopy (%) for a neighborhood."""
    if neighborhood not in HEAT_DATA:
        return {"error": f"Unknown neighborhood. Choose from: {list(HEAT_DATA)}"}
    return HEAT_DATA[neighborhood]

# Test without the model: no API call.
print(get_heat_info("Kensington"))
print(get_heat_info("Chestnut Hill"))

{'temp_c': 38.5, 'canopy': 9}
{'temp_c': 33.0, 'canopy': 48}


### Describe the tool and connect it to the model

The description tells the model what it can request. The model supplies `neighborhood`; Python looks up the data. Unlike Chat Completions, Responses puts the tool's `name` and `parameters` directly in the tool definition, without a nested `function` field.

In [24]:
# 2. DESCRIPTION: the schema sent to the model (Responses API format).
HEAT_TOOLS = [{
    "type": "function",
    "name": "get_heat_info",
    "description": "Get fictional classroom heat and tree canopy data for a Philadelphia neighborhood.",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {"neighborhood": {"type": "string"}},
        "required": ["neighborhood"],
        "additionalProperties": False,
    },
}]

# 3. MODEL: ask for the next action or the final answer.
def ask_model(history):
    return client.responses.create(
        model="gpt-5-mini",
        instructions=(
            "You are an urban-sustainability assistant. "
            "Use get_heat_info for neighborhood heat and canopy questions. "
            "The only supported neighborhoods are Kensington and Chestnut Hill. "
            "These are fictional classroom data, not real observations. "
            "Report temperature in C and canopy in percent; compare canopy using percentage points. "
            "Do not claim that two sample values prove tree canopy causes temperature differences. "
            "If a tool returns an error, ask for a supported neighborhood. Give a short answer."
        ),
        input=history,
        tools=HEAT_TOOLS,
    )

In [29]:
# 4. LOOP: request → run tool → return observation → repeat.
def run_heat_agent(question):
    history = [{"role": "user", "content": question}]
    for step in range(5):
        print(f"\n--- Heat agent turn {step + 1} ---")
        response = ask_model(history)
        # Keep ALL output items, including reasoning items needed by the API.
        history += response.output

        print("Model output------:", len(response.output), response.output)
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text
        
        for call in calls:
            arguments = json.loads(call.arguments)
            print("Tool request:", call.name, arguments)
            if call.name == "get_heat_info":
                result = get_heat_info(**arguments)
            else:
                result = {"error": f"Unknown tool: {call.name}"}
            print("Tool result:", result)
            history.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })
    raise RuntimeError("Stopped after five model turns without a final answer.")

### Run the comparison

Predict the tool calls before running: the model should request both neighborhoods. It may request them together or in separate turns. Each `function_call_output` returns Python's result; `call_id` connects it to the correct request. The model then writes an answer based on those observations.

In [30]:
heat_question = (
    "Compare the heat and tree canopy in Kensington and Chestnut Hill. "
    "Which is hotter in the sample data, and what are the temperature "
    "and canopy differences?"
)
print("Final answer:", run_heat_agent(heat_question))


--- Heat agent turn 1 ---
Model output------: 3 [ResponseReasoningItem(id='rs_04ec06703e94f23b006ae08dd1725c87d288d97838bce1d17e', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq4I3TX_f8i_dS-el7Sn7cuuhbPfpuvpoMSBtCX5fZpyime71Uyh9JSgT3xCg_BClpFci26Ni7ZkbFKLB3-G6oDSP9UivPmdn01ZyrFOqSc6JiAJIClxThglfwdukwOFpWDeg_a1BqAFeIaahW1pSdsAGqT0aAExtPFhdm1Ibvp9jZffH8SFjuJhVCIn_2Ys5KQ1Lr7Tm85wQd2Y7tfSj9a9hKBuYGbvz2LvgWLrI5rGqC0uGbJRKwpaA4IDDWK5iTvb_VELYEVA4NKJjrSdCUHCZAfEsNqeJoTp6hbA9vn3L7As9T_QnN1gj6lED-KHTigrLlv4QydrppqXRg4ZfSeT3MJHSQeQFrGc_QXMc0MeNl7aaXZj8sEzfIiXQCcDd6iS_vDzxiPAjsnN3N8sUIedSRRcMk_cfwx0HctIMt9P5CcUBsThd1wMI1e3EwlpI_c_GRZE-QUUHPKqYdnxnhi0XlivkkHR-ba0MNra-G5WGlLG5Va7kiy3NkEuBmKWCpa03lejqp_fzMx3a6ikStsR4a4KknJ9-3ceodzPJLLYXFCzmvKdU5vB9j-pJKfqFTNLDHDjUyIgTWhqNp3PP5j53fVtvjJTMDYXNz9_M07wcfvKfm5ahsp6qksLX5VKqQ4YzlQiMR8S7hX9D2Eb9yGY8K7AXeQo1MjCCTjsz0khL-TGXYL00gB-ANf8YToizIKt8kd5m7hmdIv4sqGCZB9Du59AjODl6haO5Kd4Dz6ytZNfQMmANW3ycJTfyoxQalMfCaTFb8vmETon39FV84bgB_9flyCAx

### Check the answer and explore

Using the sample table, Kensington is **5.5 C warmer** and has **39 percentage points less canopy** than Chestnut Hill. These describe the fictional sample, not actual Philadelphia conditions. Two values alone cannot establish a causal relationship.

1. Change the question to ask about Kensington only. How many tool calls are needed?
2. Ask about Fishtown. Does the agent explain that the neighborhood is not in the table?
3. Add a fictional Fishtown entry, rerun the data cell, update the supported-neighborhood instructions in `ask_model`, and ask again.
4. Change Kensington's sample temperature and rerun the data cell. Does the final answer use the updated tool result?

**What did you build?** A model that chooses a data tool, observes its result, and answers a question. You still wrote the schema, dispatcher, history handling, and loop yourself. In [building_agents_with_langgraph.ipynb](building_agents_with_langgraph.ipynb), a framework supplies much of this supporting code.

Further reading: [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling) and [GPT-5 mini](https://developers.openai.com/api/docs/models/gpt-5-mini).